# Train XAI YOLO From Scratch

Notebook nay huan luyen YOLO + XAI ngay tu dau tu checkpoint pretrained, khong fine-tune tu baseline da train xong.
Muc tieu la so sanh cong bang giua baseline `model.train(...)` va pipeline `XAI-guided training` voi cung diem khoi tao.


In [1]:
!pip install ultralytics==8.4.61

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.5 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numb

In [2]:
from pathlib import Path

REPO_URL = "https://github.com/Thanhmay2406/xai-driven-saliency-loss.git"
CLONE_DIR = Path("/kaggle/working/xai-driven-saliency-loss")

if not CLONE_DIR.exists():
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print(f"Repo already exists: {CLONE_DIR}")
    %cd {CLONE_DIR}
    !git pull --ff-only


Cloning into '/kaggle/working/xai-driven-saliency-loss'...
remote: Enumerating objects: 385, done.
remote: Counting objects: 100% (385/385), done.
remote: Compressing objects: 100% (269/269), done.
remote: Total 385 (delta 156), reused 337 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (385/385), 33.27 MiB | 40.99 MiB/s, done.
Resolving deltas: 100% (156/156), done.


In [3]:
import sys
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working")
DEFAULT_REPO_ROOT = KAGGLE_WORKING / "xai-driven-saliency-loss"
REPO_ROOT = DEFAULT_REPO_ROOT if (DEFAULT_REPO_ROOT / "src").exists() else Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

CFG = {
    "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
    "INIT_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "/kaggle/working/yolo_xai_from_scratch",
    "XAI_RUN_NAME": "xai_from_scratch_v2",
    "IMGSZ": 640,
    "BATCH": 16,
    "DEVICE": 0,
    "WORKERS": 4,
    "EPOCHS": 60,
    "PATIENCE": 10,
    "SEED": 42,
    "OPTIMIZER": "SGD",
    "LR": 1e-2,
    "LRF": 1e-2,
    "MOMENTUM": 0.937,
    "WEIGHT_DECAY": 5e-4,
    "WARMUP_EPOCHS": 3,
    "XAI_METHOD": "activation",
    "LAMBDA_SALIENCY": 0.006,
    "NUM_CLASSES": 6,
    "USE_PROGRESSIVE_LAMBDA": True,
    "PROGRESSIVE_WARMUP_EPOCHS": 30,
    "GT_MASK_MODE": "soft",
    "SOFT_MASK_SIGMA": 0.35,
    "SHRUNK_BOX_RATIO": 0.7,
}

DATASET_YAML = Path(CFG["DATASET_YAML"])
INIT_WEIGHTS = CFG["INIT_WEIGHTS"]
TRAIN_ARGS = {
    "data": str(DATASET_YAML),
    "project": CFG["PROJECT_DIR"],
    "name": CFG["XAI_RUN_NAME"],
    "epochs": CFG["EPOCHS"],
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
    "workers": CFG["WORKERS"],
    "patience": CFG["PATIENCE"],
    "seed": CFG["SEED"],
}
VAL_ARGS = {
    "data": str(DATASET_YAML),
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
}
print("Dataset YAML:", DATASET_YAML)
print("Init weights:", INIT_WEIGHTS)
print("Train args:", TRAIN_ARGS)


REPO_ROOT: /kaggle/working/xai-driven-saliency-loss
Dataset YAML: /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml
Init weights: yolo11n.pt
Train args: {'data': '/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml', 'project': '/kaggle/working/yolo_xai_from_scratch', 'name': 'xai_from_scratch_v2', 'epochs': 60, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4, 'patience': 10, 'seed': 42}


In [4]:
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

Path(CFG["PROJECT_DIR"]).mkdir(parents=True, exist_ok=True)


In [5]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Load Initial Pretrained Checkpoint

Cell nay load checkpoint pretrained ban dau (`yolo11n.pt`) de kiem tra diem xuat phat truoc khi train XAI tu dau.


In [6]:
init_model = YOLO(str(INIT_WEIGHTS))
init_model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_

In [7]:
val_results = init_model.val(
    **VAL_ARGS,
    split="val",
)
val_results


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3.3±0.4 MB/s, size: 11.0 KB)
val: Scanning /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels/valid... 1603 images, 289 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1603/1603 330.8it/s 4.8s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 7.2it/s 14.1s0.1s
                   all       1603       1959    0.00167    0.00301   3.25e-05   8.04e-06
               bicycle        293        293    0.00665     0.0102   0.000152   3.91e-05
                   car        352        409          0          0          0          0
            motorcycle       

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f1edc9eb5f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [8]:
test_results = init_model.val(
    **VAL_ARGS,
    split="test",
)
test_results


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3.0±0.9 MB/s, size: 9.9 KB)
val: Scanning /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels/test... 1562 images, 335 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1562/1562 333.6it/s 4.7s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 98/98 7.8it/s 12.5s0.1s
                   all       1562       1780    0.00036   0.000932   2.22e-06   2.22e-07
               bicycle        258        258          0          0          0          0
                   car        280        302          0          0          0          0
            motorcycle        434        449          0          0          0          0
              airplane     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f1e73cb4f80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

## XAI Training From Scratch

Pha nay huan luyen YOLO + XAI ngay tu checkpoint pretrained `yolo11n.pt`, khong di qua buoc fine-tune tu baseline da co.
De so sanh cong bang voi baseline, notebook nay giu cung `TRAIN_ARGS` va custom loop chi khac o viec cong them `saliency_loss`.
Vi XAI dang dung custom loop voi `train_loader`, early stopping phai duoc theo doi thu cong sau moi epoch.

Training trong repo hien tai phai dung `activation` de `saliency_loss` thuc su co gradient.
`Grad-CAM`/`Grad-CAM++`/`EigenCAM` chi dung cho visualization hoac evaluation offline.


In [12]:
import importlib
import inspect
import src.training as training_pkg
import src.training.yolo_xai_trainer as yolo_xai_trainer_module
import src.xai.activation_attention as activation_attention_module

importlib.reload(activation_attention_module)
importlib.reload(yolo_xai_trainer_module)
importlib.reload(training_pkg)

from src.training import (
    UltralyticsYOLOXAIDetectionTrainer,
    UltralyticsYOLOXAITrainerConfig,
    train_ultralytics_yolo_xai,
)

xai_cfg = UltralyticsYOLOXAITrainerConfig(
    xai_method=CFG["XAI_METHOD"],
    lambda_saliency=CFG["LAMBDA_SALIENCY"],
    num_classes=CFG["NUM_CLASSES"],
    use_progressive_lambda=CFG["USE_PROGRESSIVE_LAMBDA"],
    progressive_warmup_epochs=CFG["PROGRESSIVE_WARMUP_EPOCHS"],
    gt_mask_mode=CFG["GT_MASK_MODE"],
    soft_mask_sigma=CFG["SOFT_MASK_SIGMA"],
    shrunk_box_ratio=CFG["SHRUNK_BOX_RATIO"],
)
print("Trainer source:", inspect.getfile(UltralyticsYOLOXAIDetectionTrainer))
xai_cfg

Trainer source: /kaggle/working/xai-driven-saliency-loss/src/training/yolo_xai_trainer.py


UltralyticsYOLOXAITrainerConfig(xai_method='activation', lambda_saliency=0.006, num_classes=6, target_layer=None, prefer_branch='small', use_progressive_lambda=True, progressive_warmup_epochs=30, gt_mask_mode='soft', soft_mask_sigma=0.35, shrunk_box_ratio=0.7)

In [16]:
import json
import random

import numpy as np
import torch

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

train_overrides = {
    **TRAIN_ARGS,
    "model": str(INIT_WEIGHTS),
    "optimizer": CFG["OPTIMIZER"],
    "lr0": CFG["LR"],
    "lrf": CFG["LRF"],
    "momentum": CFG["MOMENTUM"],
    "weight_decay": CFG["WEIGHT_DECAY"],
    "warmup_epochs": CFG["WARMUP_EPOCHS"],
    "plots": True,
    "val": True,
}

trainer = train_ultralytics_yolo_xai(
    train_overrides=train_overrides,
    xai_config=xai_cfg,
)

run_dir = Path(trainer.save_dir)
summary_path = run_dir / "summary" / "run_summary.json"
history_path = run_dir / "weights" / "train_history.json"
print("Run dir:", run_dir)
print("Summary:", summary_path)
print("History:", history_path)

run_summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(run_summary, indent=2, ensure_ascii=False))
run_summary


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xai_from_scratch_v2-6, nbs=64, nms=False, opset=None, o

In [17]:
!rm -rf /kaggle/working/xai-driven-saliency-loss